In [1]:
print("Hello world")

Hello world


In [56]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from xgboost import XGBRegressor

In [42]:
df = pd.read_csv("NYC.csv")

In [4]:
df.head(5)

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435


In [7]:
df.shape

(1458644, 11)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1458644 entries, 0 to 1458643
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   id                  1458644 non-null  str    
 1   vendor_id           1458644 non-null  int64  
 2   pickup_datetime     1458644 non-null  str    
 3   dropoff_datetime    1458644 non-null  str    
 4   passenger_count     1458644 non-null  int64  
 5   pickup_longitude    1458644 non-null  float64
 6   pickup_latitude     1458644 non-null  float64
 7   dropoff_longitude   1458644 non-null  float64
 8   dropoff_latitude    1458644 non-null  float64
 9   store_and_fwd_flag  1458644 non-null  str    
 10  trip_duration       1458644 non-null  int64  
dtypes: float64(4), int64(3), str(4)
memory usage: 122.4 MB


In [9]:
df.isnull().sum()

id                    0
vendor_id             0
pickup_datetime       0
dropoff_datetime      0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
trip_duration         0
dtype: int64

In [43]:
df['pickup_datetime']=pd.to_datetime(df['pickup_datetime'])
df['dropoff_datetime']=pd.to_datetime(df['dropoff_datetime'])

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1458644 entries, 0 to 1458643
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   id                  1458644 non-null  str           
 1   vendor_id           1458644 non-null  int64         
 2   pickup_datetime     1458644 non-null  datetime64[us]
 3   dropoff_datetime    1458644 non-null  datetime64[us]
 4   passenger_count     1458644 non-null  int64         
 5   pickup_longitude    1458644 non-null  float64       
 6   pickup_latitude     1458644 non-null  float64       
 7   dropoff_longitude   1458644 non-null  float64       
 8   dropoff_latitude    1458644 non-null  float64       
 9   store_and_fwd_flag  1458644 non-null  str           
 10  trip_duration       1458644 non-null  int64         
dtypes: datetime64[us](2), float64(4), int64(3), str(2)
memory usage: 122.4 MB


In [44]:
df['pickup_hour']=df['pickup_datetime'].dt.hour
df['pickup_day']=df['pickup_datetime'].dt.day

In [27]:
df.head(5)

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,pickup_hour,pickup_day
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455,17,14
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663,0,12
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N,2124,11,19
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429,19,6
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435,13,26


In [47]:
X = df.drop('trip_duration', axis=1)
y = df['trip_duration']
y = np.log1p(y)

In [48]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [57]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=10),
    "XGBoost": XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1)
}

In [50]:
experiments = []

In [36]:
for name, model in models.items():
    
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])
    
    scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=5,
        scoring="r2"
    )
    
    result = {
        "Model": name,
        "CV Mean R2": scores.mean(),
        "CV Std": scores.std()
    }
    
    experiments.append(result)
    
    print("\nModel:", name)
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Std:", scores.std())

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/pipeline.py", line 613, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
  File "/Library/Frameworks/

In [45]:
num_cols = X.select_dtypes(include=[np.number]).columns
cat_cols = X.select_dtypes(exclude=[np.number]).columns

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

In [46]:
pipeline = Pipeline([("preprocessor", preprocessor),("model", model)])

In [51]:
for name, model in models.items():
    
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])
    
    scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=5,
        scoring="r2"
    )
    
    result = {
        "Model": name,
        "CV Mean R2": scores.mean(),
        "CV Std": scores.std()
    }
    
    experiments.append(result)
    
    print("\nModel:", name)
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Std:", scores.std())

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/pipeline.py", line 613, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
  File "/Library/Frameworks/

In [52]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Separate column types
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

# Preprocessing
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

/var/folders/k8/kc8jrng519q81_pvhp9qvnyr0000gn/T/ipykernel_1467/3400293869.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=["object"]).columns


In [58]:
for name, model in models.items():
    
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=5,
        scoring="r2"
    )
    
    result = {
        "Model": name,
        "CV Mean R2": scores.mean(),
        "CV Std": scores.std()
    }
    
    experiments.append(result)
    
    print("\nModel:", name)
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Std:", scores.std())


Model: Linear Regression
Scores: [0.0163101  0.04835613 0.03046317 0.03759924 0.03128945]
Mean: 0.032803615601593575
Std: 0.010437290812537345

Model: Random Forest
Scores: [0.46005623 0.45997741 0.45782192 0.45193646 0.45738536]
Mean: 0.457435476521234
Std: 0.0029570521085453346

Model: XGBoost
Scores: [0.55570367 0.57072852 0.55468234 0.55863359 0.56595301]
Mean: 0.5611402264902374
Std: 0.0062071249080212195


In [59]:
results_df = pd.DataFrame(experiments)
print("\nFinal Comparison:")
print(results_df)


Final Comparison:
               Model  CV Mean R2    CV Std
0  Linear Regression    0.032804  0.010437
1  Linear Regression    0.032804  0.010437
2      Random Forest    0.457435  0.002957
3            XGBoost    0.561140  0.006207


In [60]:
from geopy.distance import geodesic

def calculate_distance(row):
    return geodesic(
        (row["pickup_latitude"], row["pickup_longitude"]),
        (row["dropoff_latitude"], row["dropoff_longitude"])
    ).km

df["distance"] = df.apply(calculate_distance, axis=1)

In [61]:
df["pickup_weekday"] = df["pickup_datetime"].dt.weekday
df["is_weekend"] = df["pickup_weekday"].isin([5,6]).astype(int)

In [62]:
df = df[df["trip_duration"] < 10000]
df = df[df["trip_duration"] > 10]

In [63]:
df = df.drop(columns=[
    "id",
    "pickup_datetime",
    "dropoff_datetime"
], errors="ignore")

In [ ]:
X = df.drop("trip_duration", axis=1)
y = df["trip_duration"]

y = np.log1p(y)

In [67]:
X = X.drop(columns=["store_and_fwd_flag"], errors="ignore")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [69]:
print(X_train.dtypes)

vendor_id              int64
passenger_count        int64
pickup_longitude     float64
pickup_latitude      float64
dropoff_longitude    float64
dropoff_latitude     float64
pickup_hour            int32
pickup_day             int32
distance             float64
pickup_weekday         int32
is_weekend             int64
dtype: object


In [ ]:
num_cols = X_train.columns

preprocessor = ColumnTransformer([("num", StandardScaler(), num_cols)])

In [71]:
for name, model in models.items():
    
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])
    
    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=3,
        scoring="r2",
        n_jobs=-1
    )
    
    result = {
        "Model": name,
        "CV Mean R2": scores.mean(),
        "CV Std": scores.std()
    }
    
    experiments.append(result)
    
    print("\nModel:", name)
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Std:", scores.std())


Model: Linear Regression
Scores: [0.38198956 0.30220196 0.33785102]
Mean: 0.3406808486498014
Std: 0.03263455349584764

Model: Random Forest
Scores: [0.72836295 0.73113523 0.72832888]
Mean: 0.7292756877359667
Std: 0.0013149703812457107

Model: XGBoost
Scores: [0.75840603 0.75999363 0.7573012 ]
Mean: 0.7585669543065604
Std: 0.0011050555697792845


In [73]:
df["speed"] = df["distance"] / (df["trip_duration"] + 1)
df["is_rush"] = df["pickup_hour"].isin([7,8,9,17,18,19]).astype(int)
df["distance_per_passenger"] = df["distance"] / (df["passenger_count"] + 1)

In [74]:
df["time_bin"] = pd.cut(
    df["pickup_hour"],
    bins=[0,6,12,18,24],
    labels=["night","morning","afternoon","evening"]
)

In [75]:
df = df.drop(columns=["speed"])

In [76]:
num_cols = X_train.select_dtypes(include=["int64","float64"]).columns
cat_cols = X_train.select_dtypes(include=["object","category"]).columns

In [77]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [4, 6, 8],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8, 1]
}

In [78]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(n_jobs=-1))
])

In [79]:
search = RandomizedSearchCV(
    pipeline,
    param_grid,
    n_iter=5,
    cv=3,
    scoring="r2",
    n_jobs=-1
)

search.fit(X_train, y_train)

print("Best Score:", search.best_score_)
print("Best Params:", search.best_params_)

Best Score: 0.7872744940819641
Best Params: {'model__subsample': 1, 'model__n_estimators': 200, 'model__max_depth': 8, 'model__learning_rate': 0.1}


In [ ]:
from sklearn.cluster import MiniBatchKMeans

coords = np.vstack((
    df[["pickup_latitude", "pickup_longitude"]],
    df[["dropoff_latitude", "dropoff_longitude"]]
))

kmeans = MiniBatchKMeans(n_clusters=20, random_state=42, batch_size=10000)
kmeans.fit(coords)

df["pickup_cluster"] = kmeans.predict(df[["pickup_latitude", "pickup_longitude"]])
df["dropoff_cluster"] = kmeans.predict(df[["dropoff_latitude", "dropoff_longitude"]])

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but MiniBatchKMeans was fitted without feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but MiniBatchKMeans was fitted without feature names
  warnings.warn(


In [85]:
df["cluster_route"] = (
    df["pickup_cluster"].astype(str) + "_" +
    df["dropoff_cluster"].astype(str)
)

In [86]:
df["group_travel"] = (df["passenger_count"] > 1).astype(int)

In [92]:
X = df.drop("trip_duration", axis=1)
y = df["trip_duration"]

In [93]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.compose import ColumnTransformer

num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns

print("Numerical:", num_cols)
print("Categorical:", cat_cols)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

Numerical: Index(['vendor_id', 'passenger_count', 'pickup_longitude', 'pickup_latitude',
       'dropoff_longitude', 'dropoff_latitude', 'distance', 'is_weekend',
       'is_rush', 'distance_per_passenger', 'direction', 'group_travel'],
      dtype='str')
Categorical: Index(['store_and_fwd_flag', 'time_bin', 'cluster_route'], dtype='str')


/var/folders/k8/kc8jrng519q81_pvhp9qvnyr0000gn/T/ipykernel_1467/2184170616.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=["object", "category"]).columns


In [95]:
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(n_jobs=-1))
])

In [96]:
param_grid = {
    "model__n_estimators": [150, 200, 300],
    "model__max_depth": [6, 8, 10],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8, 1]
}

In [97]:
search = RandomizedSearchCV(
    pipeline,
    param_grid,
    n_iter=5,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

print("\nBest Score:", search.best_score_)
print("Best Params:", search.best_params_)


Best Score: 0.7772095004717509
Best Params: {'model__subsample': 1, 'model__n_estimators': 300, 'model__max_depth': 10, 'model__learning_rate': 0.1}


In [98]:
best_model = search.best_estimator_

test_score = best_model.score(X_test, y_test)

print("\nFinal Test R2:", test_score)


Final Test R2: 0.7782561779022217


BEST MODEL : XGBOOST WITH RANDOMIZED CV
BEST FEATURE : distance(geodesic km)
BEST TUNING : In XGBoost, keeping n_estimators to 200 and changing max depth to 8 got good accuracy.
BEST CV R2: 0.787 
Test R2 : 0.778